## 1. Общий setup

Заполните `REPO_URL`, при необходимости поменяйте `BRANCH`, затем выполните блок. CSV-файлы должны лежать в корне репозитория.

In [ ]:
# 1. Общий setup
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Timur713/HSE_RAG_Hackaton"  # pure URL or markdown link are both accepted
BRANCH = "main"
PROJECT_DIR = Path("/content/legal_hse_4")
RUN_OPTIONAL_DENSE = False  # True installs sentence-transformers and enables dense configs


def normalize_repo_url(value: str) -> str:
    value = str(value).strip()
    markdown_match = re.search(r"\((https?://[^)]+|git@[^)]+)\)", value)
    if markdown_match:
        value = markdown_match.group(1)
    else:
        plain_match = re.search(r"https?://[^\s\]]+|git@[^\s\]]+", value)
        if plain_match:
            value = plain_match.group(0)
    value = value.strip().strip("'").strip('"')
    if value.endswith("/"):
        value = value[:-1]
    return value


def run_cmd(cmd, *, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result


REPO_URL = normalize_repo_url(REPO_URL)
if not REPO_URL:
    raise ValueError("Заполните REPO_URL чистым URL GitHub-репозитория.")
print("Using repo:", REPO_URL)

# Colab workspace is disposable. Re-cloning is more reliable than trying to repair a dirty git checkout.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

clone_result = run_cmd(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=False)
if clone_result.returncode != 0:
    print(f"Clone with --branch {BRANCH!r} failed; retrying default branch clone.")
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run_cmd(["git", "clone", REPO_URL, str(PROJECT_DIR)])
    branches = run_cmd(["git", "-C", str(PROJECT_DIR), "branch", "-a"], check=False)
    if f"remotes/origin/{BRANCH}" in branches.stdout:
        run_cmd(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH])
    else:
        print(f"Branch {BRANCH!r} not found; using repository default branch.")

run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR)])
if RUN_OPTIONAL_DENSE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-optional.txt")])

os.chdir(PROJECT_DIR)
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from legal_hse.config import PathConfig

paths = PathConfig.from_root(PROJECT_DIR)
paths.ensure_dirs()
paths.require_input_files()
print(f"Project ready: {PROJECT_DIR}")


## 2. Все эксперименты

`cv` используйте для перебора конфигураций на train-pool. `holdout` запускайте отдельно только для финальной проверки выбранных экспериментов.

In [ ]:
# 2a. Focused sweep: токенизация + лемматизация для sparse retrieval
import sys
from legal_hse.experiments import default_experiments

# pymorphy3 нужен только для lemmatize=True; без него код fallback'ится в обычные токены.
run_cmd([sys.executable, "-m", "pip", "install", "-q", "pymorphy3>=2.0.6"])

TOKENIZATION_EXPERIMENT_NAMES = [
    # controls from the current strong sparse baseline
    "bm25_doc",
    "bm25_chunk_line_10_5_max",
    "tfidf_char_doc_3_5",
    "rrf_sparse_doc_chunk_char",
    # pure lemmatization checks
    "tfidf_word_lemma_doc",
    "bm25_lemma_doc",
    # legal-aware tokenizer: keeps ст. 333.19, 75-ФЗ, НК/ГК/РФ and removes ФИО/адрес noise
    "tfidf_word_legal_lemma_doc",
    "bm25_legal_lemma_doc",
    "bm25_legal_lemma_chunk_line_10_5_max",
    "rrf_legal_lemma_doc_chunk",
    "rrf_sparse_legal_lemma_char",
    "rrf_sparse_legal_lemma_word_char",
    # phrase variant for stable legal collocations; keep only if validation beats simpler lemmas
    "bm25_legal_phrase_doc",
    "bm25_legal_phrase_chunk_line_10_5_max",
    "rrf_legal_phrase_doc_chunk",
]

available_experiments = {spec.name for spec in default_experiments(include_optional=RUN_OPTIONAL_DENSE)}
missing = sorted(set(TOKENIZATION_EXPERIMENT_NAMES) - available_experiments)
if missing:
    raise ValueError(f"Unknown experiments: {missing}")

MODE = "cv"
EXPERIMENT_NAMES = TOKENIZATION_EXPERIMENT_NAMES
SEED = 42
N_SPLITS = 5
print("Tokenization/lemmatization sweep:")
for name in EXPERIMENT_NAMES:
    print("-", name)


In [ ]:
# 2b. Recall@20/30/50/100 diagnostics + candidate-generation sweep (single-cell run)
# This cell runs the full pre-rerank recall workflow:
# 1) oracle diagnostics for current branches;
# 2) deeper chunk retrieval;
# 3) chunk-view sweep;
# 4) BM25 k1/b sweep;
# 5) quota/union fusion;
# 6) optional dense candidate generation.

import sys

from legal_hse.experiments import (
    recall_candidate_experiments,
    run_recall_oracle_diagnostics,
    run_suite,
    select_best_experiment,
)

# pymorphy3 is required for the lemmatized sparse configs to be reproducible in Colab.
run_cmd([sys.executable, "-m", "pip", "install", "-q", "pymorphy3>=2.0.6"])

# Dense is enabled by default for this recall sweep because point 5 is part of the plan.
# Set ENABLE_RECALL_DENSE = False before running this cell to run sparse-only.
# Set ENABLE_BGE_M3 = True before running this cell to add BGE-M3 experiments too.
ENABLE_RECALL_DENSE = bool(globals().get("ENABLE_RECALL_DENSE", True))
ENABLE_BGE_M3 = bool(globals().get("ENABLE_BGE_M3", False))
if ENABLE_RECALL_DENSE and not RUN_OPTIONAL_DENSE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-optional.txt")])
    RUN_OPTIONAL_DENSE = True

RECALL_MODE = globals().get("RECALL_MODE", "cv")  # "cv", "holdout", or "train"
RECALL_SEED = globals().get("RECALL_SEED", 42)
RECALL_N_SPLITS = globals().get("RECALL_N_SPLITS", 5)
RECALL_EVAL_KS = tuple(globals().get("RECALL_EVAL_KS", (5, 10, 20, 30, 50, 100)))
RECALL_EVAL_DEPTH = max(globals().get("RECALL_EVAL_DEPTH", 100), max(RECALL_EVAL_KS))
BASELINE_FUSION = "rrf_sparse_legal_lemma_char"

EXTRA_RECALL_EXPERIMENTS = recall_candidate_experiments(
    include_optional=ENABLE_RECALL_DENSE,
    include_bge_m3=ENABLE_BGE_M3,
)
RECALL_EXPERIMENT_NAMES = [BASELINE_FUSION] + [spec.name for spec in EXTRA_RECALL_EXPERIMENTS]

print("Step 1/2: oracle diagnostics for the current sparse candidate branches")
RECALL_DIAGNOSTICS = run_recall_oracle_diagnostics(
    data_dir=PROJECT_DIR,
    branch_names=[
        "bm25_legal_lemma_doc",
        "bm25_legal_lemma_chunk_line_10_5_max",
        "tfidf_char_doc_3_5",
    ],
    baseline_fusion_name=BASELINE_FUSION,
    mode=RECALL_MODE,
    include_optional=ENABLE_RECALL_DENSE,
    extra_experiments=EXTRA_RECALL_EXPERIMENTS,
    seed=RECALL_SEED,
    n_splits=RECALL_N_SPLITS,
    depth=RECALL_EVAL_DEPTH,
    eval_ks=RECALL_EVAL_KS,
)
display(RECALL_DIAGNOSTICS)

print(f"Step 2/2: running {len(RECALL_EXPERIMENT_NAMES)} recall experiments at k={RECALL_EVAL_KS}")
summary = run_suite(
    data_dir=PROJECT_DIR,
    output_dir=PROJECT_DIR / "reports",
    experiment_names=RECALL_EXPERIMENT_NAMES,
    extra_experiments=EXTRA_RECALL_EXPERIMENTS,
    mode=RECALL_MODE,
    include_optional=ENABLE_RECALL_DENSE,
    seed=RECALL_SEED,
    n_splits=RECALL_N_SPLITS,
    eval_depth=RECALL_EVAL_DEPTH,
    eval_ks=RECALL_EVAL_KS,
    comparison_baseline=BASELINE_FUSION,
)

cols = [
    "experiment",
    "status",
    "priority",
    "n_splits",
    "valid_recall@5_mean",
    "valid_recall@10_mean",
    "valid_recall@20_mean",
    "valid_recall@30_mean",
    "valid_recall@50_mean",
    "valid_recall@100_mean",
    "valid_recall@5_delta_vs_baseline",
    "valid_recall@5_wins_vs_baseline",
    "valid_recall@5_losses_vs_baseline",
    "holdout_recall@5_mean",
    "holdout_recall@10_mean",
    "holdout_recall@20_mean",
    "holdout_recall@30_mean",
    "holdout_recall@50_mean",
    "holdout_recall@100_mean",
    "duration_sec",
]
visible_cols = [c for c in cols if c in summary.columns]
sort_col = next(
    c
    for c in [
        "holdout_recall@50_mean",
        "valid_recall@50_mean",
        "holdout_recall@100_mean",
        "valid_recall@100_mean",
        "holdout_recall@30_mean",
        "valid_recall@30_mean",
        "holdout_recall@20_mean",
        "valid_recall@20_mean",
        "holdout_recall@5_mean",
        "valid_recall@5_mean",
    ]
    if c in summary.columns
)
ranked_summary = summary[visible_cols].sort_values(["status", sort_col], ascending=[True, False])
display(ranked_summary.head(40))

BEST_RECALL50_EXPERIMENT = select_best_experiment(summary, metric=sort_col)
BEST_RECALL30_EXPERIMENT = select_best_experiment(
    summary,
    metric=next(c for c in ["holdout_recall@30_mean", "valid_recall@30_mean", sort_col] if c in summary.columns),
)
BEST_RECALL20_EXPERIMENT = select_best_experiment(
    summary,
    metric=next(c for c in ["holdout_recall@20_mean", "valid_recall@20_mean", sort_col] if c in summary.columns),
)
BEST_RECALL100_EXPERIMENT = select_best_experiment(
    summary,
    metric=next(c for c in ["holdout_recall@100_mean", "valid_recall@100_mean", sort_col] if c in summary.columns),
)
BEST_EXPERIMENT = select_best_experiment(summary)  # Keeps the submission cell Recall@5-oriented by default.

print("BEST_RECALL100_EXPERIMENT =", BEST_RECALL100_EXPERIMENT)
print("BEST_RECALL50_EXPERIMENT =", BEST_RECALL50_EXPERIMENT)
print("BEST_RECALL30_EXPERIMENT =", BEST_RECALL30_EXPERIMENT)
print("BEST_RECALL20_EXPERIMENT =", BEST_RECALL20_EXPERIMENT)
print("BEST_EXPERIMENT [Recall@5 default] =", BEST_EXPERIMENT)
print("Full summary saved under:", PROJECT_DIR / "reports")

## 2c. Отдельный BGE-M3 retrieval experiment

Эта ячейка проверяет `BAAI/bge-m3` отдельно от общего sweep: dense retrieval через `sentence-transformers`, native dense+sparse retrieval через `FlagEmbedding`, и fusion с текущими сильными sparse-ветками.


In [ ]:
# 2c. Separate BGE-M3 retrieval experiment
# Default parameters are chosen for Colab GPU and this corpus:
# - line chunks 10/5 are the strongest current passage view and keep the index small (~2k chunks);
# - max_length=2048 limits very long legal lines without paying full 8192-token cost;
# - native BGE-M3 uses dense+sparse weights 0.7/0.3, then chunk->doc max aggregation;
# - strict CV remains the default, holdout should be used only after selecting 1-2 candidates.

import sys
from dataclasses import replace

from legal_hse.experiments import recall_candidate_experiments, run_suite, select_best_experiment

run_cmd([sys.executable, "-m", "pip", "install", "-q", "pymorphy3>=2.0.6"])
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-optional.txt")])

BGE_M3_MODE = globals().get("BGE_M3_MODE", "cv")  # "cv", "holdout", or "train"
BGE_M3_SEED = globals().get("BGE_M3_SEED", 42)
BGE_M3_N_SPLITS = globals().get("BGE_M3_N_SPLITS", 5)
BGE_M3_EVAL_KS = tuple(globals().get("BGE_M3_EVAL_KS", (5, 10, 20, 30, 50, 100)))
BGE_M3_EVAL_DEPTH = max(globals().get("BGE_M3_EVAL_DEPTH", 100), max(BGE_M3_EVAL_KS))
BGE_M3_BATCH_SIZE = int(globals().get("BGE_M3_BATCH_SIZE", 4))
BGE_M3_ST_BATCH_SIZE = int(globals().get("BGE_M3_ST_BATCH_SIZE", 8))
BGE_M3_MAX_LENGTH = int(globals().get("BGE_M3_MAX_LENGTH", 2048))
BGE_M3_RUN_CHAR = bool(globals().get("BGE_M3_RUN_CHAR", False))
BGE_M3_BASELINE = "rrf_sparse_deep_legal_lemma_char"

_extra_specs = recall_candidate_experiments(include_optional=False, include_bge_m3=True)
BGE_M3_EXTRA_EXPERIMENTS = []
for spec in _extra_specs:
    params = spec.params
    if spec.kind == "bge_m3_chunk":
        params = {
            **spec.params,
            "config": {
                **spec.params.get("config", {}),
                "batch_size": BGE_M3_BATCH_SIZE,
                "max_length": BGE_M3_MAX_LENGTH,
            },
        }
    elif spec.name == "dense_bge_m3_chunk_line_10_5_rd600":
        params = {
            **spec.params,
            "config": {
                **spec.params.get("config", {}),
                "batch_size": BGE_M3_ST_BATCH_SIZE,
                "max_seq_length": BGE_M3_MAX_LENGTH,
            },
        }
    BGE_M3_EXTRA_EXPERIMENTS.append(replace(spec, params=params))

BGE_M3_EXPERIMENT_NAMES = [
    BGE_M3_BASELINE,
    "dense_bge_m3_chunk_line_10_5_rd600",
    "bge_m3_dense_sparse_chunk_line_10_5_rd600",
    "rrf_sparse_bge_m3_line",
    "rrf_sparse_bge_m3_native_line",
    "quota_sparse_bge_m3_native_line_q8",
]
if BGE_M3_RUN_CHAR:
    BGE_M3_EXPERIMENT_NAMES.append("bge_m3_dense_sparse_chunk_char_1600_800_rd600")

print(f"Running {len(BGE_M3_EXPERIMENT_NAMES)} BGE-M3 experiments in mode={BGE_M3_MODE}")
for name in BGE_M3_EXPERIMENT_NAMES:
    print("-", name)

bge_m3_summary = run_suite(
    data_dir=PROJECT_DIR,
    output_dir=PROJECT_DIR / "reports",
    experiment_names=BGE_M3_EXPERIMENT_NAMES,
    extra_experiments=BGE_M3_EXTRA_EXPERIMENTS,
    mode=BGE_M3_MODE,
    include_optional=False,
    seed=BGE_M3_SEED,
    n_splits=BGE_M3_N_SPLITS,
    eval_depth=BGE_M3_EVAL_DEPTH,
    eval_ks=BGE_M3_EVAL_KS,
    comparison_baseline=BGE_M3_BASELINE,
)

bge_cols = [
    "experiment",
    "status",
    "priority",
    "n_splits",
    "valid_recall@5_mean",
    "valid_recall@10_mean",
    "valid_recall@20_mean",
    "valid_recall@50_mean",
    "valid_recall@100_mean",
    "valid_recall@5_delta_vs_baseline",
    "valid_recall@5_wins_vs_baseline",
    "valid_recall@5_losses_vs_baseline",
    "holdout_recall@5_mean",
    "holdout_recall@10_mean",
    "holdout_recall@20_mean",
    "holdout_recall@50_mean",
    "holdout_recall@100_mean",
]
bge_cols = [col for col in bge_cols if col in bge_m3_summary.columns]
display(bge_m3_summary[bge_cols].sort_values(bge_cols[4] if len(bge_cols) > 4 else "experiment", ascending=False))
print("Best BGE-M3 run:", select_best_experiment(bge_m3_summary))


In [ ]:
# 2. Все эксперименты
from legal_hse.experiments import run_suite, select_best_experiment

MODE = globals().get("MODE", "cv")  # "holdout", "cv" or "train"
EXPERIMENT_NAMES = globals().get("EXPERIMENT_NAMES", None)  # None runs default suite
SEED = globals().get("SEED", 42)
N_SPLITS = globals().get("N_SPLITS", 5)

summary = run_suite(
    data_dir=PROJECT_DIR,
    output_dir=PROJECT_DIR / "reports",
    experiment_names=EXPERIMENT_NAMES,
    mode=MODE,
    include_optional=RUN_OPTIONAL_DENSE,
    seed=SEED,
    n_splits=N_SPLITS,
    eval_depth=50,
)

cols = [
    "experiment",
    "status",
    "n_splits",
    "train_recall@5_mean",
    "train_recall@5_std",
    "valid_recall@5_mean",
    "valid_recall@5_micro",
    "valid_recall@5_se",
    "valid_recall@5_std",
    "valid_recall@5_delta_vs_baseline",
    "valid_recall@5_wins_vs_baseline",
    "valid_recall@5_losses_vs_baseline",
    "valid_recall@10_mean",
    "valid_recall@20_mean",
    "valid_recall@50_mean",
    "holdout_recall@5_mean",
    "holdout_recall@5_micro",
    "holdout_recall@5_se",
    "holdout_recall@5_std",
    "holdout_recall@5_delta_vs_baseline",
    "holdout_recall@5_wins_vs_baseline",
    "holdout_recall@5_losses_vs_baseline",
    "holdout_recall@10_mean",
    "holdout_recall@20_mean",
    "holdout_recall@50_mean",
]
visible_cols = [c for c in cols if c in summary.columns]
sort_col = next(c for c in ["holdout_recall@5_mean", "valid_recall@5_mean", "train_recall@5_mean"] if c in summary.columns)
display(summary[visible_cols].sort_values(["status", sort_col], ascending=[True, False]))
BEST_EXPERIMENT = select_best_experiment(summary)
print("BEST_EXPERIMENT =", BEST_EXPERIMENT)

## 3. Загрузка результатов на GitHub

Введите `GITHUB_USERNAME`, `GIT_EMAIL`, `SSH_PRIVATE_KEY_B64`. Ключ должен иметь write-доступ к репозиторию.

In [ ]:
# 3. Загрузка результатов [метрик] на GitHub
import base64
import getpass
import os
import subprocess
from pathlib import Path

GITHUB_USERNAME = input("GITHUB_USERNAME: ").strip()
GIT_EMAIL = input("GIT_EMAIL: ").strip()
SSH_PRIVATE_KEY_B64 = getpass.getpass("SSH_PRIVATE_KEY_B64: ").strip()
SSH_REMOTE_URL = ""  # optional: git@github.com:<user>/<repo>.git

ssh_dir = Path.home() / ".ssh"
ssh_dir.mkdir(mode=0o700, exist_ok=True)
key_path = ssh_dir / "id_ed25519"
key_path.write_bytes(base64.b64decode(SSH_PRIVATE_KEY_B64))
key_path.chmod(0o600)
subprocess.run(["ssh-keyscan", "github.com"], stdout=(ssh_dir / "known_hosts").open("a"), check=True)

subprocess.run(["git", "config", "user.name", GITHUB_USERNAME], check=True)
subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)

if not SSH_REMOTE_URL and REPO_URL.startswith("https://github.com/"):
    repo_path = REPO_URL.split("https://github.com/", 1)[1]
    repo_path = repo_path[:-4] if repo_path.endswith(".git") else repo_path
    SSH_REMOTE_URL = f"git@github.com:{repo_path}.git"
if SSH_REMOTE_URL:
    subprocess.run(["git", "remote", "set-url", "origin", SSH_REMOTE_URL], check=True)

paths_to_add = ["reports/metrics", "reports/summary_latest.csv"]
paths_to_add += [str(path) for path in Path("reports").glob("summary_*.csv")]
subprocess.run(["git", "add", *paths_to_add], check=True)
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, check=True).stdout.strip()
if status:
    subprocess.run(["git", "commit", "-m", "Add retrieval experiment metrics"], check=True)
    subprocess.run(["git", "push", "origin", f"HEAD:{BRANCH}"], check=True)
    print("Metrics pushed to GitHub.")
else:
    print("No metric changes to push.")

## 4. Формирование submission файла

По умолчанию используется лучший experiment name из блока 2. Его можно заменить вручную.

In [ ]:
# 4. Формирование submission файла
import pandas as pd
from legal_hse.experiments import create_submission

FINAL_EXPERIMENT = globals().get("BEST_EXPERIMENT", "rrf_bm25_doc_chunk")
SUBMISSION_PATH = create_submission(
    data_dir=PROJECT_DIR,
    experiment_name=FINAL_EXPERIMENT,
    output_path=PROJECT_DIR / "submissions" / f"submission_{FINAL_EXPERIMENT}.csv",
    include_optional=RUN_OPTIONAL_DENSE,
    top_k=5,
)
print("Submission:", SUBMISSION_PATH)
display(pd.read_csv(SUBMISSION_PATH).head(15))